# Beschreibung: 
## Vorsicht: Durch das Ausführen geht der Verbrauch von Resourcen sehr stark nach oben.

# Importe:

In [2]:
import sys
import os

import pandas as pd
import numpy as np

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, repo_root)

from rst_functions import discretize, indiscernibility, dependency, quick_reduct, induce_rules, compute_coverage

print(f"Pfad zu allen Daten: {repo_root} \nPfad dieser Datei: {os.getcwd()}")

Pfad zu allen Daten: /home/samel/01. Projekte/01. Master/COMPARE_RST 
Pfad dieser Datei: /home/samel/01. Projekte/01. Master/COMPARE_RST/Manuelle_Ausfuehrungen/Crimes_Prediction


# Daten laden:
Heart Failure Prediction Dataset: https://www.kaggle.com/datasets/utkarsh1093/crime-data-from-2020-to-nov2025?select=Crime_Data_from_2020_to_Present.parquet

In [4]:
crimes_copy = pd.read_parquet(f"{repo_root}/Daten/Crimes/LA_Crimes_2025.parquet")

In [3]:
crimes = crimes_copy.copy()
# crimes.to_csv()  # Falls es man die Datei als .csv haben möchte.
print(crimes.shape)
print(crimes.columns)

(1004991, 28)
Index(['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME',
       'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes',
       'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc',
       'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1',
       'LOCATION', 'LAT', 'LON', 'occ_year', 'occ_month', 'occ_date',
       'occ_day'],
      dtype='object')


# Ausführung:

### Vorbereitung: (Datenaufbereitung)

In [4]:
# Als Entscheidung wird folgendes genutzt:
decision_attr = "Weapon Used Cd" # Wenn nach den Umsänden gefragt, wo es Wahrscheinlich ist eine Waffe zu nutzen, oder 

In [5]:
cutoffs = {
    #"TIME OCC": [20, 22], # Aufteilung in 18/19, 20/21, 22/23/24
    #"TIME OCC": [20, 22], # Aufteilung in 18/19, 20/21, 22/23/24
}

# Diskretisierung der numerischen Daten:
X = crimes.drop(columns=[decision_attr])

# ALLE Konditionsattribute diskretisieren (inkl. kategoriale)
crimes_disc = discretize(X, bins=4, cutoffs=cutoffs)

# Entscheidungsattribut wieder anhängen
crimes_disc[decision_attr] = crimes[decision_attr]
print(crimes_disc.columns)

Index(['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME',
       'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes',
       'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc',
       'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1', 'LOCATION', 'LAT',
       'LON', 'occ_year', 'occ_month', 'occ_date', 'occ_day', 'DR_NO_disc',
       'AREA_disc', 'Rpt Dist No_disc', 'Part 1-2_disc', 'Crm Cd_disc',
       'Vict Age_disc', 'Premis Cd_disc', 'Crm Cd 1_disc', 'LAT_disc',
       'LON_disc', 'Weapon Used Cd'],
      dtype='object')


In [6]:
cond_attrs = []
for col in crimes_disc.columns:
    if col != decision_attr:
        # diskretisierte numerische Werte
        if col.endswith("_disc"):
            cond_attrs.append(col)
        # direkt kategorische Werte
        elif crimes_disc[col].dtype == "object":
            cond_attrs.append(col)
            
cond_attrs.remove('Weapon Desc') # Das ist ein Identifier, daher muss es raus.
cond_attrs.remove('DR_NO_disc') # Das ist ein Identifier, daher muss es raus.
cond_attrs.remove('TIME OCC') # Das ist ein Identifier, daher muss es raus.
cond_attrs.remove('LAT_disc') # Ist ein kein reiner Identifier, aber wird durch den Ort abgedeckt.
cond_attrs.remove('LON_disc') # Ist ein kein reiner Identifier, aber wird durch den Ort abgedeckt.
cond_attrs.remove('Mocodes') # Ist ein kein reiner Identifier, aber wird durch den Ort abgedeckt.
cond_attrs.remove('Status Desc') # Ist ein kein reiner Identifier, aber wird durch den Ort abgedeckt.
cond_attrs.remove('Crm Cd_disc') # Ist ein kein reiner Identifier, aber wird durch den Ort abgedeckt.
cond_attrs.remove('Crm Cd Desc') # Ist ein kein reiner Identifier, aber wird durch den Ort abgedeckt.
cond_attrs.remove('Crm Cd 1_disc') # Ist ein kein reiner Identifier, aber wird durch den Ort abgedeckt.

cond_attrs.remove('Status') # Ist ein kein reiner Identifier, aber wird durch den Ort abgedeckt.

print(cond_attrs)

['AREA NAME', 'Vict Sex', 'Vict Descent', 'Premis Desc', 'LOCATION', 'occ_month', 'occ_day', 'AREA_disc', 'Rpt Dist No_disc', 'Part 1-2_disc', 'Vict Age_disc', 'Premis Cd_disc']


### Datenbetrachtung:

In [7]:
# Redukte
reduct, info = quick_reduct(crimes_disc, cond_attrs, decision_attr)

γ(C) mit allen Attributen: 0.989688
Einzel-γ-Werte:
  AREA NAME: γ = 0.000000
  Vict Sex: γ = 0.000000
  Vict Descent: γ = 0.000002
  Premis Desc: γ = 0.000051
  LOCATION: γ = 0.078564
  occ_month: γ = 0.000000
  occ_day: γ = 0.000000
  AREA_disc: γ = 0.000000
  Rpt Dist No_disc: γ = 0.000000
  Part 1-2_disc: γ = 0.000000
  Vict Age_disc: γ = 0.000000
  Premis Cd_disc: γ = 0.000000

Mindestens ein Attribut hat γ({a}) > 0 → benutze quick_reduct_monotone.


In [8]:
# Regeln
rules = induce_rules(crimes_disc, reduct, decision_attr)

# Resultate

In [9]:
print("Ergebnisse:")

print(f"\n{decision_attr}:\n{reduct}")
print(f"\nAnzahl Regeln: {len(rules)}\n")

for r in rules[:10]:
    print(r)

#print("Rules:", rules_pass_biased[2]) # Einzeln
#print("Rules:", rules_pass_biased) # Das wären alle

Ergebnisse:

Weapon Used Cd:
['LOCATION', 'occ_month', 'occ_day', 'Premis Desc', 'Vict Age_disc', 'Vict Descent', 'Part 1-2_disc', 'Vict Sex', 'AREA NAME', 'Premis Cd_disc', 'AREA_disc', 'Rpt Dist No_disc']

Anzahl Regeln: 963370

{'premise': {'LOCATION': '7800    BEEMAN                       AV', 'occ_month': 'Nov', 'occ_day': 'Sat', 'Premis Desc': 'SINGLE FAMILY DWELLING', 'Vict Age_disc': np.int64(2), 'Vict Descent': 'Hispanic/Latin/Mexican', 'Part 1-2_disc': np.int64(2), 'Vict Sex': 'Male', 'AREA NAME': 'N Hollywood', 'Premis Cd_disc': np.int64(2), 'AREA_disc': np.int64(2), 'Rpt Dist No_disc': np.int64(2)}, 'decision': np.float64(0.0), 'support': 1}
{'premise': {'LOCATION': '7800    BEEMAN                       AV', 'occ_month': 'Nov', 'occ_day': 'Tue', 'Premis Desc': 'STREET', 'Vict Age_disc': np.int64(3), 'Vict Descent': 'Hispanic/Latin/Mexican', 'Part 1-2_disc': np.int64(1), 'Vict Sex': 'Male', 'AREA NAME': 'N Hollywood', 'Premis Cd_disc': np.int64(0), 'AREA_disc': np.int64(2), 